<a href="https://colab.research.google.com/github/VenkataBhanuTejaKonijeti/Deepfake-Detection-using-EfficientNetB6-AGSK/blob/main/code_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os
import shutil
import random
import pandas as pd
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import drive

# ================================
# 1️⃣ MOUNT GOOGLE DRIVE & SET PATHS
# ================================
drive.mount('/content/drive', force_remount=True)  # Ensure remount
dataset_zip = "/content/drive/MyDrive/dataset.zip"
extract_path = "/content/dataset_extracted"

# ================================
# 2️⃣ EXTRACT THE DATASET
# ================================
if not os.path.exists(extract_path):
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
print("✅ Dataset extracted successfully!")

# ================================
# 3️⃣ ORGANIZE IMAGES INTO 'REAL' & 'FAKE' FOLDERS
# ================================
train_real_folder = os.path.join(extract_path, "training_real")
train_fake_folder = os.path.join(extract_path, "training_fake")
train_path = os.path.join(extract_path, "train")
test_path = os.path.join(extract_path, "test")

for category in ["real", "fake"]:
    os.makedirs(os.path.join(train_path, category), exist_ok=True)
    os.makedirs(os.path.join(test_path, category), exist_ok=True)

def move_images(source_folder, dest_folder):
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
    for image_name in os.listdir(source_folder):
        if any(image_name.lower().endswith(ext) for ext in valid_extensions):
            shutil.move(os.path.join(source_folder, image_name), os.path.join(dest_folder, image_name))

move_images(train_real_folder, os.path.join(train_path, "real"))
move_images(train_fake_folder, os.path.join(train_path, "fake"))

# ================================
# 4️⃣ SPLIT DATASET INTO TRAIN & TEST (80-20)
# ================================
def split_data(source_folder, train_dest, test_dest, split_ratio=0.8):
    files = os.listdir(source_folder)
    random.shuffle(files)
    split_index = int(len(files) * split_ratio)
    for f in files[split_index:]:
        shutil.move(os.path.join(source_folder, f), os.path.join(test_dest, f))

split_data(os.path.join(train_path, "real"), os.path.join(train_path, "real"), os.path.join(test_path, "real"))
split_data(os.path.join(train_path, "fake"), os.path.join(train_path, "fake"), os.path.join(test_path, "fake"))

# ================================
# 5️⃣ LOAD DATASET USING PYTORCH
# ================================
transform = transforms.Compose([
    transforms.Resize((380, 380)),  # Increased input size
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2),
    transforms.RandomAffine(degrees=15, translate=(0.15, 0.15)),
    transforms.RandomResizedCrop(380, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = datasets.ImageFolder(root=train_path, transform=transform)
test_dataset = datasets.ImageFolder(root=test_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

# ================================
# 6️⃣ DEFINE MODEL (EfficientNet-B6 + AGSK)
# ================================
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # Debug mode
torch.backends.cuda.matmul.allow_tf32 = False  # Ensure precision
torch.cuda.set_device(0)  # Ensure the correct GPU is used

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
efficientnet_b6 = timm.create_model("tf_efficientnet_b6_ns", pretrained=True, num_classes=512)  # Ensure better adaptation

class AGSK_EfficientNetB6(nn.Module):
    def __init__(self, base_model):
        super(AGSK_EfficientNetB6, self).__init__()
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-2])
        self.agsk = nn.Conv2d(2304, 2304, kernel_size=3, padding=1, groups=2, bias=False)
        self.fc = nn.Sequential(
            nn.Linear(2304, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.mean(dim=[2, 3])
        x = self.fc(x)
        return x

model = AGSK_EfficientNetB6(efficientnet_b6).to(device)
print(f"✅ Model moved to {device}")

# ================================
# 7️⃣ TRAIN THE MODEL
# ================================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

num_epochs = 100

torch.cuda.empty_cache()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}, Accuracy: {accuracy:.4f}")

torch.save(model.state_dict(), "efficientnet_b6_agsk.pth")
print("✅ Training completed and model saved!")
# ================================
# 8️⃣ EVALUATE THE MODEL ON TEST DATA
# ================================
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
        outputs = model(images)
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total
print(f"✅ Test Accuracy: {test_accuracy:.4f}")